In [ ]:
from pynq import Overlay, MMIO
import time

BIT = "/home/xilinx/jupyter_notebooks/pokemon_game/pokemon_game_block.bit"
overlay = Overlay(BIT)

ip_name = [k for k in overlay.ip_dict.keys() if "pokemon_game" in k.lower()][0]
base = overlay.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

CTRL = 0x00
S0   = 0x04  # state_in[31:0]
S1   = 0x08
S2   = 0x0C
S3   = 0x10  # state_in[127:96]

O0   = 0x14  # state_out[31:0]
O1   = 0x18
O2   = 0x1C
O3   = 0x20  # state_out[127:96]

def start_and_wait():
    # Clear DONE + START in one AXI transaction (bit1 + bit0)
    mmio.write(CTRL, 0x3)

    # Prove the write stuck: bit0 is slv_reg[0][0] and should read back as 1
    ctrl_rb = mmio.read(CTRL)
    print("CTRL readback after start:", hex(ctrl_rb))

    for _ in range(200000):
        if (mmio.read(CTRL) >> 1) & 1:   # DONE is bit1 (done_r)
            return
    raise RuntimeError("Timed out waiting for DONE")

# write a known 128-bit pattern
mmio.write(S0, 0x11111111)
mmio.write(S1, 0x22222222)
mmio.write(S2, 0x33333333)
mmio.write(S3, 0x44444444)

start_and_wait()

out = [mmio.read(O0), mmio.read(O1), mmio.read(O2), mmio.read(O3)]
print([hex(x) for x in out])

In [ ]:
print("Chosen IP:", ip_name, "base:", hex(base))

# Write a pattern into STATE_IN regs then read them back
mmio.write(0x04, 0x11111111)
mmio.write(0x08, 0x22222222)
mmio.write(0x0C, 0x33333333)
mmio.write(0x10, 0x44444444)

print("Readback S0..S3:",
      hex(mmio.read(0x04)),
      hex(mmio.read(0x08)),
      hex(mmio.read(0x0C)),
      hex(mmio.read(0x10)))

# Now try START once and immediately print CTRL (raw)
mmio.write(0x00, 0x2)  # clear
mmio.write(0x00, 0x1)  # start
print("CTRL after start:", hex(mmio.read(0x00)))

In [ ]:
# write input
mmio.write(0x04, 0x11111111)
mmio.write(0x08, 0x22222222)
mmio.write(0x0C, 0x33333333)
mmio.write(0x10, 0x44444444)

# start
mmio.write(0x00, 0x2)  # clear
mmio.write(0x00, 0x1)  # start

# read outputs
print([hex(mmio.read(a)) for a in (0x14,0x18,0x1C,0x20)])
print("CTRL:", hex(mmio.read(0x00)))

In [ ]:
print(ip_name)
print(overlay.ip_dict[ip_name])
print("CTRL:", hex(mmio.read(CTRL)))

In [ ]:
from pynq import Overlay
ol = Overlay("/home/xilinx/jupyter_notebooks/pokemon_game/battle_block.bit")
print([k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()])
ip = [k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()][0]
print(ip)
print(ol.ip_dict[ip]["parameters"])

In [1]:
from pynq import Overlay, MMIO
import time

BIT = "/home/xilinx/jupyter_notebooks/pokemon_game/battle_block.bit"
ol = Overlay(BIT)

ip_name = [k for k in ol.ip_dict.keys() if "pokemon" in k.lower() or "battle" in k.lower()][0]
base = ol.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

CTRL=0x00
S0=0x04; S1=0x08; S2=0x0C; S3=0x10
O0=0x14; O1=0x18; O2=0x1C; O3=0x20

def start_and_wait():
    mmio.write(CTRL, 0x2)      # clear done (bit1)
    mmio.write(CTRL, 0x1)      # start (bit0)
    for i in range(200000):
        if (mmio.read(CTRL) >> 1) & 1:
            return True
    return False

mmio.write(S0, 0x11111111)
mmio.write(S1, 0x22222222)
mmio.write(S2, 0x33333333)
mmio.write(S3, 0x44444444)

ok = start_and_wait()
print("done:", ok, "CTRL:", hex(mmio.read(CTRL)))
print("OUT:", [hex(mmio.read(x)) for x in (O0,O1,O2,O3)])

done: True CTRL: 0x3
OUT: ['0x11111111', '0x22222222', '0x33333333', '0x44444444']


In [2]:
from pynq import Overlay, MMIO

BIT = "/home/xilinx/jupyter_notebooks/pokemon_game/battle_block.bit"
ol = Overlay(BIT, download=True)   # force re-download
ol.download()                      # extra explicit

ip_name = [k for k in ol.ip_dict.keys() if "battle" in k.lower()][0]
base = ol.ip_dict[ip_name]["phys_addr"]
mmio = MMIO(base, 0x1000)

print("ADDR_WIDTH:", ol.ip_dict[ip_name]["parameters"]["ADDR_WIDTH"])
print("WIZ_NUM_REG:", ol.ip_dict[ip_name]["parameters"]["WIZ_NUM_REG"])

ADDR_WIDTH: 6
WIZ_NUM_REG: 9


In [3]:
def dump_regs(tag):
    print(tag, [hex(mmio.read(4*i)) for i in range(9)])

dump_regs("BEFORE")

# write inputs
mmio.write(0x04, 0x11111111)  # reg1
mmio.write(0x08, 0x22222222)  # reg2
mmio.write(0x0C, 0x33333333)  # reg3
mmio.write(0x10, 0x44444444)  # reg4

dump_regs("AFTER_IN")

# start
mmio.write(0x00, 0x2)  # clear done
mmio.write(0x00, 0x1)  # start

dump_regs("AFTER_START")

BEFORE ['0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0']
AFTER_IN ['0x0', '0x11111111', '0x22222222', '0x33333333', '0x44444444', '0x0', '0x0', '0x0', '0x0']
AFTER_START ['0x3', '0x11111111', '0x22222222', '0x33333333', '0x44444444', '0x11111111', '0x22222222', '0x33333333', '0x44444444']


In [2]:
def dump_regs(tag):
    print(tag, [hex(mmio.read(4*i)) for i in range(9)])

dump_regs("AFTER")

AFTER ['0x3', '0x11111111', '0x22222222', '0x33333333', '0x44444444', '0x11111111', '0x22222222', '0x33333333', '0x44444444']


In [ ]:
import time
import random
from decimal import Decimal
import boto3

from pynq import Overlay, MMIO
from ipywidgets import Button, VBox, HBox, Output, SelectMultiple, Label
from IPython.display import display, clear_output

# =============================================================
# CONFIG
# =============================================================

TABLE_NAME = "BattleTable"
GAME_ID    = "GAME_0001"

TURN_TIMEOUT_MS   = 15000
SHIELD_TIMEOUT_MS = 10000

# Overlay + MMIO
overlay = Overlay("pokemon_game.bit")   # <-- adjust if needed
IP_BASE = 0x43C00000                    # <-- adjust if needed
mmio    = MMIO(IP_BASE, 0x1000)

CTRL      = 0x00
STATE_IN  = [0x04, 0x08, 0x0C, 0x10]
STATE_OUT = [0x14, 0x18, 0x1C, 0x20]

# Actions (must match battle_engine)
ACTION_ATTACK  = 0
ACTION_SPECIAL = 1
ACTION_SWITCH  = 2
ACTION_RESOLVE = 3

POKEMON_STATS = {
    0: {"name":"Pikachu",    "hp":100, "atk":15, "special":40, "cost":50},
    1: {"name":"Charmander", "hp":110, "atk":18, "special":45, "cost":60},
    2: {"name":"Squirtle",   "hp":120, "atk":12, "special":35, "cost":40},
    3: {"name":"Bulbasaur",  "hp":105, "atk":14, "special":38, "cost":45},
}

# DynamoDB
dynamodb = boto3.resource("dynamodb")
table    = dynamodb.Table(TABLE_NAME)

# =============================================================
# UTIL
# =============================================================

def now_ms() -> int:
    return int(time.time() * 1000)

def deep_intify(x):
    # Convert boto3 Decimals into ints everywhere (critical for bit ops)
    if isinstance(x, Decimal):
        return int(x)
    if isinstance(x, dict):
        return {k: deep_intify(v) for k, v in x.items()}
    if isinstance(x, list):
        return [deep_intify(v) for v in x]
    return x

def fmt_state128_hex(state_int: int) -> str:
    return "0x%032X" % (state_int & ((1<<128)-1))

# =============================================================
# DB IO
# =============================================================

def load_game():
    t0 = time.perf_counter_ns()
    r  = table.get_item(Key={"GameID": GAME_ID})
    t1 = time.perf_counter_ns()
    item = deep_intify(r["Item"])
    return item, (t1 - t0)

def save_game(item):
    # timestamps
    ms = now_ms()
    if item.get("created_at_ms", 0) == 0:
        item["created_at_ms"] = ms
    item["updated_at_ms"] = ms

    # packed state cache (optional debug)
    try:
        state_int = pack_state(item)
        item["packed_state_128"] = fmt_state128_hex(state_int)
    except Exception:
        # if pack fails during TEAM_SELECT, keep whatever is there
        pass

    t0 = time.perf_counter_ns()
    table.put_item(Item=item)
    t1 = time.perf_counter_ns()
    return (t1 - t0)

# =============================================================
# 128-bit PACK/UNPACK
# =============================================================

def pack_state(item) -> int:
    # Globals
    state = 0
    state |= (item["game_over"] & 1) << 127
    state |= (item["winner"]    & 3) << 125
    state |= (item["player_turn"] & 1) << 124

    ps = item["pending_special"]
    state |= (ps["active"] & 1) << 123
    state |= (ps["attacker"] & 1) << 122
    state |= (ps["shield_used"] & 1) << 121

    def pack_player(p_battle) -> int:
        block = 0
        block |= (p_battle["shields"] & 3) << 56
        block |= (p_battle["active_index"] & 3) << 54
        for i, slot in enumerate(p_battle["slots"]):
            base = 36 - i*18
            block |= (slot["pokemon_id"] & 3) << (base + 16)
            block |= (slot["hp"] & 0xFF) << (base + 8)
            block |= (slot["energy"] & 0xFF) << base
        return block

    p0 = item["players"]["0"]["battle"]
    p1 = item["players"]["1"]["battle"]

    state |= pack_player(p0) << 60
    state |= pack_player(p1) << 0

    return state

def unpack_state(state: int, item):
    item["game_over"]   = (state >> 127) & 1
    item["winner"]      = (state >> 125) & 3
    item["player_turn"] = (state >> 124) & 1

    ps = item["pending_special"]
    ps["active"]      = (state >> 123) & 1
    ps["attacker"]    = (state >> 122) & 1
    ps["shield_used"] = (state >> 121) & 1

    def unpack_player(off_bits, p_battle):
        block = (state >> off_bits) & ((1<<60)-1)
        p_battle["shields"]      = (block >> 56) & 3
        p_battle["active_index"] = (block >> 54) & 3

        for i in range(3):
            base = 36 - i*18
            slot = p_battle["slots"][i]
            slot["pokemon_id"] = (block >> (base + 16)) & 3
            slot["hp"]         = (block >> (base + 8)) & 0xFF
            slot["energy"]     = (block >> base) & 0xFF

    unpack_player(60, item["players"]["0"]["battle"])
    unpack_player(0,  item["players"]["1"]["battle"])

# =============================================================
# FPGA CALL
# =============================================================

def run_fpga(state: int, action: int, switch_idx: int, this_player: int):
    t0 = time.perf_counter_ns()

    # write 128-bit state_in
    for i in range(4):
        mmio.write(STATE_IN[i], (state >> (32*i)) & 0xFFFFFFFF)

    ctrl = ((action & 3) << 2) | ((switch_idx & 3) << 4) | ((this_player & 1) << 8)

    # start pulse
    mmio.write(CTRL, ctrl | 1)

    # poll done (CTRL bit1)
    while ((mmio.read(CTRL) >> 1) & 1) == 0:
        pass

    # read 128-bit output
    out = 0
    for i in range(4):
        out |= (mmio.read(STATE_OUT[i]) & 0xFFFFFFFF) << (32*i)

    # clear done
    mmio.write(CTRL, ctrl | (1<<1))

    t1 = time.perf_counter_ns()
    return out, (t1 - t0)

# =============================================================
# GAME INIT / FLOW HELPERS
# =============================================================

def both_locked(item) -> bool:
    return (item["players"]["0"]["team_select"]["locked"] == 1 and
            item["players"]["1"]["team_select"]["locked"] == 1)

def init_battle_from_team_select(item):
    # coin flip
    item["player_turn"] = random.randint(0, 1)
    item["turn_started_at_ms"] = now_ms()

    # clear pending special
    ps = item["pending_special"]
    ps["active"] = 0
    ps["attacker"] = 0
    ps["shield_used"] = 0
    ps["defender_responded"] = 0
    ps["timestamp_ms"] = 0

    # init players
    for pid in ["0","1"]:
        chosen = item["players"][pid]["team_select"]["chosen_ids"]
        battle = item["players"][pid]["battle"]

        battle["shields"] = 2
        battle["active_index"] = 0

        for i in range(3):
            pokemon_id = int(chosen[i])
            battle["slots"][i]["pokemon_id"] = pokemon_id
            battle["slots"][i]["hp"] = POKEMON_STATS[pokemon_id]["hp"]
            battle["slots"][i]["energy"] = 0

    item["phase"] = "IN_BATTLE"
    item["game_over"] = 0
    item["winner"] = 0

def alive_bench_slots(item, pid: str):
    battle = item["players"][pid]["battle"]
    active = battle["active_index"]
    res = []
    for i,s in enumerate(battle["slots"]):
        if i != active and s["hp"] > 0:
            res.append(i)
    return res

def active_energy_and_cost(item, pid: str):
    battle = item["players"][pid]["battle"]
    aidx = battle["active_index"]
    slot = battle["slots"][aidx]
    pid_ = slot["pokemon_id"]
    en   = slot["energy"]
    cost = POKEMON_STATS[pid_]["cost"]
    return en, cost

# =============================================================
# TIMEOUTS
# =============================================================

def apply_timeouts_if_needed(item):
    ms = now_ms()

    if item["game_over"] == 1 or item["phase"] == "GAME_OVER":
        return False, None

    # Turn timeout (auto-attack) only in IN_BATTLE
    if item["phase"] == "IN_BATTLE":
        if (ms - item.get("turn_started_at_ms", 0)) >= TURN_TIMEOUT_MS:
            # auto-attack by current turn player
            return True, ("AUTO_ATTACK", item["player_turn"], ACTION_ATTACK, 0)

    # Shield timeout (auto "no shield") in PENDING_SPECIAL
    if item["phase"] == "PENDING_SPECIAL":
        ps = item["pending_special"]
        if ps["active"] == 1 and ps["defender_responded"] == 0:
            if (ms - ps.get("timestamp_ms", 0)) >= SHIELD_TIMEOUT_MS:
                # auto no-shield decision
                ps["shield_used"] = 0
                ps["defender_responded"] = 1
                return True, ("AUTO_NO_SHIELD", None, None, None)

    return False, None

# =============================================================
# UI
# =============================================================

out = Output()

title = Label("Pokémon FPGA Battle (single-board test)")

refresh_btn = Button(description="Refresh", button_style="info")

# Team selection widgets
p0_select = SelectMultiple(
    options=[(f"{POKEMON_STATS[i]['name']} ({i})", i) for i in range(4)],
    description="P0 picks",
)
p1_select = SelectMultiple(
    options=[(f"{POKEMON_STATS[i]['name']} ({i})", i) for i in range(4)],
    description="P1 picks",
)

lock_p0_btn = Button(description="Lock P0 Team", button_style="warning")
lock_p1_btn = Button(description="Lock P1 Team", button_style="warning")

# Battle buttons
attack_btn  = Button(description="Attack", button_style="success")
special_btn = Button(description="Special", button_style="success")

resolve_btn = Button(description="Resolve Special", button_style="danger")

shield_btn    = Button(description="Use Shield", button_style="warning")
no_shield_btn = Button(description="No Shield", button_style="warning")

# Switch buttons created dynamically
switch_box = HBox([])

controls_box = VBox([])

def print_state(item):
    print(f"\nGameID: {GAME_ID} | phase={item['phase']} | turn=P{item['player_turn']} | game_over={item['game_over']} winner={item['winner']}")
    ps = item["pending_special"]
    print(f"pending_special: active={ps['active']} attacker=P{ps['attacker']} shield_used={ps['shield_used']} defender_responded={ps['defender_responded']}")

    for pid in ["0","1"]:
        battle = item["players"][pid]["battle"]
        print(f"\nPlayer {pid}: shields={battle['shields']} active={battle['active_index']}")
        for i,s in enumerate(battle["slots"]):
            name = POKEMON_STATS[s["pokemon_id"]]["name"]
            tag = "<ACTIVE>" if i == battle["active_index"] else ""
            faint = "(fainted)" if s["hp"] == 0 else ""
            print(f"  Slot{i}: {name} id={s['pokemon_id']} HP={s['hp']} EN={s['energy']} {faint} {tag}")

def render(item):
    # Build the correct controls for current phase
    children = []

    if item["phase"] == "TEAM_SELECT":
        children.append(Label("TEAM_SELECT: Choose 3 Pokémon for each player, then lock both teams."))

        # lock status
        p0_locked = item["players"]["0"]["team_select"]["locked"]
        p1_locked = item["players"]["1"]["team_select"]["locked"]

        lock_p0_btn.disabled = bool(p0_locked)
        lock_p1_btn.disabled = bool(p1_locked)

        children.append(HBox([p0_select, lock_p0_btn]))
        children.append(HBox([p1_select, lock_p1_btn]))

        if both_locked(item):
            children.append(Label("Both teams locked. Starting battle automatically..."))

        controls_box.children = children
        return

    if item["phase"] == "GAME_OVER" or item["game_over"] == 1:
        children.append(Label("GAME OVER"))
        controls_box.children = children
        return

    # PENDING_SPECIAL phase
    if item["phase"] == "PENDING_SPECIAL":
        ps = item["pending_special"]
        attacker = ps["attacker"]
        defender = 1 - attacker

        if ps["defender_responded"] == 0:
            # Defender must respond
            children.append(Label(f"PENDING_SPECIAL: Defender is P{defender}. Choose shield or not."))

            # Enable shield only if defender has shields > 0
            d_shields = item["players"][str(defender)]["battle"]["shields"]
            shield_btn.disabled = (d_shields == 0)

            children.append(HBox([shield_btn, no_shield_btn]))
        else:
            # Attacker can resolve
            children.append(Label(f"PENDING_SPECIAL: Attacker is P{attacker}. Click Resolve Special."))
            children.append(resolve_btn)

        controls_box.children = children
        return

    # IN_BATTLE phase
    if item["phase"] == "IN_BATTLE":
        turn = item["player_turn"]
        children.append(Label(f"IN_BATTLE: It is Player {turn}'s turn."))

        # Special availability
        en, cost = active_energy_and_cost(item, str(turn))
        special_btn.disabled = not (en >= cost)

        # Switch availability
        bench = alive_bench_slots(item, str(turn))
        switch_buttons = []
        for idx in bench:
            b = Button(description=f"Switch → Slot{idx}", button_style="primary")
            def make_handler(target_idx):
                return lambda _b: do_fpga_action(ACTION_SWITCH, target_idx)
            b.on_click(make_handler(idx))
            switch_buttons.append(b)
        switch_box.children = switch_buttons

        children.append(HBox([attack_btn, special_btn]))
        if len(switch_buttons) > 0:
            children.append(Label("Switch options (alive bench only):"))
            children.append(switch_box)

        controls_box.children = children
        return

def do_fpga_action(action, switch_idx=0):
    with out:
        clear_output()

        item, db_read_ns = load_game()

        # timeouts first (so UI doesn't fight stale state)
        fired, info = apply_timeouts_if_needed(item)
        if fired and info and info[0] in ("AUTO_NO_SHIELD",):
            # already applied in-memory; save and refresh
            db_write_ns = save_game(item)
            print("Applied timeout:", info[0])
            print_state(item)
            print(f"\nLatency: db_read={db_read_ns} ns, db_write={db_write_ns} ns")
            refresh()
            return

        # Only allow FPGA actions in IN_BATTLE or RESOLVE in PENDING_SPECIAL
        # (UI should already gate this)
        state_in = pack_state(item)

        # this_player is the current turn owner for safety
        this_player = item["player_turn"]

        state_out, hw_ns = run_fpga(state_in, action, switch_idx, this_player)

        # Apply FPGA result into DB item
        unpack_state(state_out, item)

        # Phase management around special flow
        ms = now_ms()

        if item["game_over"] == 1:
            item["phase"] = "GAME_OVER"

        else:
            # If we declared special and FPGA set pending_special=1, enter PENDING_SPECIAL
            if action == ACTION_SPECIAL and item["pending_special"]["active"] == 1:
                item["phase"] = "PENDING_SPECIAL"
                item["pending_special"]["shield_used"] = 0
                item["pending_special"]["defender_responded"] = 0
                item["pending_special"]["timestamp_ms"] = ms

            # If we resolved special and FPGA cleared pending_special, return to IN_BATTLE
            if action == ACTION_RESOLVE and item["pending_special"]["active"] == 0:
                item["phase"] = "IN_BATTLE"
                item["pending_special"]["defender_responded"] = 0
                item["pending_special"]["timestamp_ms"] = 0
                item["pending_special"]["shield_used"] = 0
                item["turn_started_at_ms"] = ms

            # For normal attack / switch, turn flipped → restart turn timer
            if action in (ACTION_ATTACK, ACTION_SWITCH):
                item["turn_started_at_ms"] = ms

        # Save
        db_write_ns = save_game(item)

        # Print + refresh
        print_state(item)
        e2e_ns = (db_read_ns + hw_ns + db_write_ns)  # approx; real e2e would include python overhead too
        print(f"\nLatency (approx): db_read={db_read_ns} ns, hw={hw_ns} ns, db_write={db_write_ns} ns, sum~={e2e_ns} ns")

    refresh()

def set_defender_shield(use_shield: int):
    with out:
        clear_output()
        item, db_read_ns = load_game()

        if item["phase"] != "PENDING_SPECIAL":
            print("Not in PENDING_SPECIAL.")
            refresh()
            return

        ps = item["pending_special"]
        attacker = ps["attacker"]
        defender = 1 - attacker

        # Defender can only use shield if they have shields > 0
        if use_shield == 1:
            d_shields = item["players"][str(defender)]["battle"]["shields"]
            if d_shields == 0:
                use_shield = 0

        ps["shield_used"] = 1 if use_shield else 0
        ps["defender_responded"] = 1

        db_write_ns = save_game(item)
        print_state(item)
        print(f"\nShield decision saved. db_read={db_read_ns} ns, db_write={db_write_ns} ns")

    refresh()

def refresh(_btn=None):
    item, _ = load_game()

    # auto-init if both locked
    if item["phase"] == "TEAM_SELECT" and both_locked(item):
        init_battle_from_team_select(item)
        save_game(item)
        item, _ = load_game()

    # apply timeouts if needed (may change state)
    fired, info = apply_timeouts_if_needed(item)
    if fired and info:
        if info[0] == "AUTO_ATTACK":
            # run FPGA auto attack
            _, _, a, sw = info
            do_fpga_action(a, sw)
            return
        if info[0] == "AUTO_NO_SHIELD":
            save_game(item)
            item, _ = load_game()

    with out:
        clear_output()
        print_state(item)

    render(item)

# Button wiring
refresh_btn.on_click(refresh)
attack_btn.on_click(lambda _b: do_fpga_action(ACTION_ATTACK, 0))
special_btn.on_click(lambda _b: do_fpga_action(ACTION_SPECIAL, 0))
resolve_btn.on_click(lambda _b: do_fpga_action(ACTION_RESOLVE, 0))

shield_btn.on_click(lambda _b: set_defender_shield(1))
no_shield_btn.on_click(lambda _b: set_defender_shield(0))

def lock_team(pid: str, picked):
    with out:
        clear_output()
        item, _ = load_game()

        # Must pick exactly 3 distinct
        picked = list(picked)
        if len(picked) != 3 or len(set(picked)) != 3:
            print("Pick exactly 3 distinct Pokémon.")
            refresh()
            return

        item["players"][pid]["team_select"]["chosen_ids"] = [int(x) for x in picked]
        item["players"][pid]["team_select"]["locked"] = 1
        save_game(item)

        print(f"Locked Player {pid} team:", picked)

    refresh()

lock_p0_btn.on_click(lambda _b: lock_team("0", p0_select.value))
lock_p1_btn.on_click(lambda _b: lock_team("1", p1_select.value))

# Show UI
display(VBox([title, refresh_btn, out, controls_box]))
refresh()